# 04: Model Training & Hyperparameter Tuning

**Objective**: Train models, tune hyperparameters, save checkpoints

**Outputs**: Trained models, hyperparameter tuning results, training curves

In [ ]:
# Setup
import os
from dotenv import load_dotenv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import joblib
import warnings
warnings.filterwarnings('ignore')

load_dotenv()
SEED = int(os.getenv('RANDOM_SEED', 42))
np.random.seed(SEED)

print(f"Configuration loaded from .env")

## Load & Prepare Data

In [ ]:
# Load preprocessed data from notebook 02
processed_path = os.getenv('PROCESSED_DATA_PATH', './data/processed')

# For demo, create sample data
n_samples = 1000
X = np.random.randn(n_samples, 10)
y = np.random.choice([0, 1], n_samples)

# Train-validation-test split
test_size = float(os.getenv('TEST_SPLIT', 0.2))
val_size = float(os.getenv('VALIDATION_SPLIT', 0.1))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=SEED, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=val_size/(1-test_size), random_state=SEED, stratify=y_train
)

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set: {X_test.shape}")

## Hyperparameter Tuning

In [ ]:
# Define hyperparameter grid
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 7, 10],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 0.9, 1.0]
}

# Base model
base_model = XGBClassifier(
    random_state=SEED,
    eval_metric='logloss'
)

# Grid search with cross-validation
print("Starting hyperparameter tuning...")
grid_search = GridSearchCV(
    base_model,
    param_grid,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=1
)

# Note: This is commented to avoid long running time
# Uncomment to run full grid search
# grid_search.fit(X_train, y_train)
# print(f"\nBest parameters: {grid_search.best_params_}")
# print(f"Best CV score: {grid_search.best_score_:.4f}")

## Train Final Model

In [ ]:
# Use best parameters (or defaults for demo)
best_params = {
    'n_estimators': 200,
    'max_depth': 7,
    'learning_rate': 0.05,
    'subsample': 0.8
}

# Train model
model = XGBClassifier(
    random_state=SEED,
    eval_metric='logloss',
    **best_params
)

# Train with early stopping
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    early_stopping_rounds=10,
    verbose=False
)

print("Model training complete!")

## Evaluate on Validation Set

In [ ]:
# Predictions
y_val_pred = model.predict(X_val)
y_val_pred_proba = model.predict_proba(X_val)[:, 1]

# Metrics
val_accuracy = accuracy_score(y_val, y_val_pred)
val_f1 = f1_score(y_val, y_val_pred)
val_auc = roc_auc_score(y_val, y_val_pred_proba)

print("Validation Metrics:")
print(f"  Accuracy: {val_accuracy:.4f}")
print(f"  F1-Score: {val_f1:.4f}")
print(f"  AUC-ROC: {val_auc:.4f}")

## Feature Importance

In [ ]:
# Feature importance
feature_importance = model.feature_importances_
feature_names = [f'Feature_{i}' for i in range(X_train.shape[1])]

# Plot
indices = np.argsort(feature_importance)[-10:]  # Top 10
plt.figure(figsize=(10, 6))
plt.barh(range(len(indices)), feature_importance[indices])
plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
plt.title('Top 10 Feature Importance')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig(os.path.join(os.getenv('FIGURES_PATH', './results/figures'), '04_feature_importance.png'), dpi=300, bbox_inches='tight')
plt.show()

## Save Model Checkpoint

In [ ]:
# Save model
model_path = os.path.join(os.getenv('MODEL_SAVE_PATH', './results/models/'), 'best_model.pkl')
os.makedirs(os.path.dirname(model_path), exist_ok=True)
joblib.dump(model, model_path)
print(f"Model saved to: {model_path}")